In [106]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [107]:
import pandas as pd
import numpy as np
#import seaborn as sns
import matplotlib.pyplot as plt

from itertools import chain

from monte_carlo_return import (
    generate_bet_return,
    compute_objective_via_simulation,
    minimize_simulation,
    INDEX_TO_SCENARIO
)
from analytical_return import (
    expectation,
    second_moment,
    variance,
    compute_objective_via_analytical,
    softmax,
)
from data import (
    load_metadata_artefacts,
    load_odds,
    join_metadata,
    build_empty_dataframe,
    apply_final_treatment,
)
from GameProbs import GameProbs
from dependencies.utils import get_scenarios
from filter import filter_by_linear_combination
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

In [108]:
metadata, gameid_to_outcome = load_metadata_artefacts("data/metadata-with-date-new.parquet")
odds = load_odds("data/odds-new-correct.parquet")
odds = join_metadata(odds, metadata)
print(metadata.shape)
print(odds.shape)

(17311, 9)
(2113785, 10)


In [109]:
from data import load_map
data = load_map("data/meanSurface-new.json")

In [110]:
GAME_ID = "4698337" # Chapecoense x Flamengo
my_game = GameProbs(GAME_ID) 
df = my_game.build_dataframe()
df

,0,1,2,3,4,5,6
0,0.0409,0.0921,0.1113,0.0938,0.0619,0.0340,0.0270
1,0.0264,0.0656,0.0818,0.0718,0.0491,0.0279,0.0232
2,0.0097,0.0249,0.0334,0.0294,0.0206,0.0120,0.0103
3,0.0026,0.0069,0.0093,0.0090,0.0061,0.0036,0.0032
4,0.0006,0.0015,0.0021,0.0020,0.0016,0.0009,0.0008
5,0.0001,0.0003,0.0004,0.0004,0.0003,0.0002,0.0002
6,0.0000,0.0001,0.0001,0.0001,0.0001,0.0000,0.0001


In [111]:
odds = odds[odds.Datetime.apply(str)=='2022-07-09']

In [112]:
odds.GameId.unique()

array(['5925201', '5925200', '5925186'], dtype=object)

In [113]:
GAME_ID = '5925201'
odds_sample = odds[(odds.GameId==GAME_ID)]
#odds_sample = join_metadata(odds_sample, metadata)
games_ids = odds_sample['GameId'].unique()

# Initialize dict to store dataframes of favorable bet opportunities
odds_dict = {}
# Initialize dict to store 7x7 matrices/dataframes of real probabilities 
df_probs_dict = {}

for game_id in games_ids:
    df = GameProbs(game_id).build_dataframe()
    odds_sample = apply_final_treatment(df_odds=odds_sample, df_real_prob=df)
    odds_sample = filter_by_linear_combination(odds_sample, n=5)
    odds_dict[game_id] = odds_sample
    df_probs_dict[game_id] = df


In [114]:
games_ids

array(['5925201'], dtype=object)

In [115]:
sum(list(chain(*df.values)))

0.9998999999999998

In [116]:
gameid_to_outcome[GAME_ID]

'4 : 0'

In [117]:
odds_sample = odds_sample[odds_sample.Market.isin(['spread', 'over/under', 'h2h', 'exact', 'both_score'])].reset_index(drop=True)
print(odds_sample.shape)

(5, 17)


In [118]:
n = len(odds_sample)
allocation_array = np.round(np.array(((1/n), ) * n), 4)
allocation_array

array([0.2, 0.2, 0.2, 0.2, 0.2])

In [119]:
#allocation_array = solution
financial_return_array = generate_bet_return(df_prob=df,
                                             df_bet=odds_sample,
                                             num_simulations=10000,
                                             allocation_array=allocation_array)
financial_return_array                                             

array([0.708, 0.708, 0.448, ..., 0.708, 1.128, 1.128])

In [120]:
print(np.mean(financial_return_array))
print(np.std(financial_return_array))

1.9644840000000001
12.776215621448475


In [121]:
odds_favorable = np.array(odds_sample['Odd'])
real_prob_favorable = np.array(odds_sample['real_prob'])
#scenario_favorable = np.array(odds_sample_favorable['Bet'])
event_favorable = list(odds_sample['BetMap'].values)
games_ids = np.array(odds_sample['GameId'])

In [122]:
games_ids

array(['5925201', '5925201', '5925201', '5925201', '5925201'],
      dtype=object)

In [123]:
compute_objective_via_simulation(
    x=allocation_array,
    df_prob=df,
    df_bet=odds_sample,
    num_simulations=10000
)

output: 0.15390422480472513


-0.15390422480472513

In [124]:
from Optimizer import Optimizer

In [125]:
solution, time_limit_flag = Optimizer().run_optimization(
    fun=compute_objective_via_simulation,
    x0=np.zeros(len(odds_sample)),
    args=(df, odds_sample, 1000)
)

/Users/marcosbarbosa/Statistics/Master/Semestre 4/soccer-betting-strategy/monte_carlo_return.py:76: RuntimeWarning: invalid value encountered in scalar divide
  output = np.mean(bet_returns) / np.std(bet_returns)
/Users/marcosbarbosa/Statistics/Master/Semestre 4/soccer-betting-strategy/monte_carlo_return.py:76: RuntimeWarning: invalid value encountered in scalar divide
  output = np.mean(bet_returns) / np.std(bet_returns)


output: 0
output: 0
output: 4.661252270288826
output: 4.502954113099627
output: 5.174724898753343
output: 4.524526836774133
output: -0.9615133481629967
output: 1.0597909726346355
output: 5.382961177324534
output: 5.1510287525886245
output: 3.8106164850222277
output: -0.3652235302994242
output: 2.1555460700428193
output: 4.776330294705997
output: 5.070243755207494
output: 4.224021624838298
output: 4.960445470069335
output: 5.345511502142959
output: 5.491551296373152
output: 5.437458940262779
output: 0.08844619018187158
output: -0.07628909566750404
output: -0.061782189505225404
output: 0.14371323094329527
output: 0.2067101416023253
output: 0.27729764088256903
output: 1.8112287139335066
output: 0.37712271533926717
output: 1.9079762736132628
output: 4.652617045359976
output: 3.878799374808429
output: 5.3476864152275505
output: 5.444495202615597
output: 5.297311275729045
output: 5.120360059251638
output: 2.282905428202629
output: -0.3357126751649937
output: 1.1365634490812533
output: 3.4302

/Users/marcosbarbosa/Statistics/Master/Semestre 4/soccer-betting-strategy/Optimizer.py:28: TookTooLong: Terminating optimization: time limit reached
  warnings.warn("Terminating optimization: time limit reached",


output: 5.49014013422494
output: 5.618900022301075
output: 5.114052666029766
output: 5.500164760628591
output: 5.677657434390338
output: 4.610440591580814
output: 5.419901732545544
output: 5.3291419408122005
output: 5.267080782292458
output: 5.2513490906347995
output: 5.349074424093655
output: 5.241192261554226
output: 5.404223085908565
output: 5.527938405333311
output: 5.466345899684559
output: 5.507066285065991
output: 5.579432071959775
output: 5.101546803172005
output: 5.207578860405633
output: 5.06585083458611
output: 5.028154057446184
output: 5.213919798838498
output: 5.218688759021712
output: 5.182985776521452
output: 5.239575145760865
output: 4.987372987633196
output: 5.228836043876527
output: 5.273965282765021
output: 5.240816044731126
output: 5.6217832909019085
output: 5.415505704100103
output: 5.228453345443059
output: 5.435728323414452
output: 5.069807576713325
output: 5.63661079300735
output: 5.047580726146883
output: 5.334168727469613
output: 5.185860898704232
output: 5.77

/Users/marcosbarbosa/Statistics/Master/Semestre 4/soccer-betting-strategy/Optimizer.py:28: TookTooLong: Terminating optimization: time limit reached
  warnings.warn("Terminating optimization: time limit reached",


In [126]:
compute_objective_via_analytical(
    x=allocation_array,
    public_odd=odds_favorable,
    real_probabilities=real_prob_favorable,
    event=event_favorable,
    games_ids=games_ids,
    df_probs_dict=df_probs_dict,
)

-0.15412574880187427

In [127]:
solution, time_limit_flag = Optimizer().run_optimization(
    fun=compute_objective_via_analytical,
    x0=np.zeros(len(odds_favorable)),
    args=(odds_favorable, real_prob_favorable, event_favorable, games_ids, df_probs_dict)
)
solution, time_limit_flag

Optimization terminated successfully.
         Current function value: -5.312544
         Iterations: 3
         Function evaluations: 136


(array([  3.21863078,   2.20755579,   0.23575249, -31.4208154 ,
        -32.96618895]),
 False)